# HalluciGuard Model Training and Evaluation Report

This notebook provides a concise, professional summary of the training and evaluation pipeline for the HalluciGuard model. All code and results are reproduced from the existing project files.

## Overview

The HalluciGuard system consists of a fine‑tuned large language model using LoRA (Low‑Rank Adaptation). The model was trained on a custom JSONL dataset and evaluated across several domains such as general knowledge, healthcare, cybersecurity, AI research, and finance.

## Training Details

* **Base model**: `Qwen/Qwen2.5-1.5B-Instruct`
* **Training script**: [`train_lora.py`](file:///c:/Users/S.Manjunath%20Reddy/OneDrive/Music/Pictures/Videos/HalluciGuard/agents/corrector_agent/training/train_lora.py)
* **Dataset**: JSONL files located at `agents/corrector_agent/training/data/` (train.jsonl and val.jsonl)
* **LoRA configuration**:
  * rank (r): 16
  * alpha: 32
  * dropout: 0.05
  * target modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`
* **Training arguments** (via `SFTConfig`):
  * per‑device train batch size: 1
  * gradient accumulation steps: 16 (4 for smoke test)
  * optimizer: `paged_adamw_8bit` on GPU, `adamw_torch` on CPU
  * epochs: 3 (1 for smoke test)
  * learning rate: 2e‑4
  * max length: 1500 tokens
  * gradient checkpointing enabled
  * mixed‑precision: bf16 when supported, otherwise fp16
* **Output directory**: `./training/qwen1.5b-corrector-lora` (or `…-smoke` for quick test)
* **Hardware**: GPU with CUDA when available; falls back to CPU otherwise.

## Training Environment

The training script automatically detects the compute environment:
* If a CUDA‑compatible GPU is present, the model is loaded in 4‑bit quantized mode using `bitsandbytes` and trained with mixed‑precision bf16 (or fp16).
* If GPU loading fails, the script gracefully falls back to CPU loading (float32).
* Peak VRAM usage is reported at the end of training.

## Evaluation Results

The evaluation was performed using the `benchmark_results.json` file generated after inference on a held‑out test set. The following cell loads and displays the key metrics.

In [ ]:
import json, pathlib, pandas as pd

# Load the benchmark JSON file
benchmark_path = pathlib.Path('benchmark_results.json')
with benchmark_path.open('r', encoding='utf-8') as f:
    data = json.load(f)

metrics = data['metrics']
results = data['results']

# Display overall metrics
display(pd.Series(metrics).rename('Overall Metrics'))

# Convert detailed results to a DataFrame for inspection
df = pd.DataFrame(results)
display(df.head())

## Detailed Metrics per Domain

The benchmark includes per‑domain accuracy figures. The following cell extracts and visualises them.

In [ ]:
domains = metrics.get('domains', {})
if domains:
    domain_df = pd.DataFrame.from_dict(domains, orient='index')
    display(domain_df)
else:
    print('No domain metrics found.')